# 05. Modelo SparkML multiclase

Entrena Logistic Regression, **SparkXGBClassifier** y One-vs-Rest LinearSVC sobre la matriz muestra × genes (200 genes seleccionados en 04). Usa el mismo split por paciente (`refined_split_pacientes`) y ponderacion por clase en los tres modelos. El XGBoost es el modelo final.

In [5]:
%run ./00_configuracion_local.py

PROJECT_DIR: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga
RAW_DIR: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\raw
RAW_METADATA_FILE: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\raw\metadata\metadatos_tcga_oficial_18_clases.csv
RAW_RNASEQ_PATH: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\raw\rnaseq
LOCAL_DATA_DIR: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local
Spark: 4.1.2


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    LogisticRegression,
    LinearSVC,
    OneVsRest
)
from xgboost.spark import SparkXGBClassifier
from xgboost.collective import Config as XGBCollectiveConfig
from pyspark.ml.functions import vector_to_array

import time
import itertools
import gc
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score, average_precision_score
)
from sklearn.preprocessing import label_binarize

# ============================================================
# Parámetros locales del experimento SparkML
# ============================================================

MODO_PRUEBA_LOCAL = True

# Número máximo de genes usados por SparkML.
# En local conviene dejarlo bajo para que no se demore demasiado.
MAX_GENES_SPARKML = 200

print("MODO_PRUEBA_LOCAL:", MODO_PRUEBA_LOCAL)
print("MAX_GENES_SPARKML:", MAX_GENES_SPARKML)


MODO_PRUEBA_LOCAL: True
MAX_GENES_SPARKML: 200


In [7]:
df_long = read_trusted(TRUSTED_LONG_PATH)
df_samples = read_trusted(TRUSTED_SAMPLES_PATH)
df_fs = load_spark_table("refined_rfe_feature_importances")
print("Genes cargados desde refined_rfe_feature_importances")

def detectar_columna_gen(df):
    for c in ["gene_id_base", "gene", "feature", "variable", "gen"]:
        if c in df.columns:
            return c
    raise ValueError(f"No se encontró columna de gen. Columnas: {df.columns}")

col_gen = detectar_columna_gen(df_fs)

if "importance" in df_fs.columns:
    df_genes_ordenados = (
        df_fs.select(F.col(col_gen).alias("gene_id_base"), F.col("importance").cast("double").alias("importance"))
        .dropna(subset=["gene_id_base"])
        .dropDuplicates(["gene_id_base"])
        .orderBy(F.desc("importance"))
    )
else:
    df_genes_ordenados = (
        df_fs.select(F.col(col_gen).alias("gene_id_base"))
        .dropna(subset=["gene_id_base"])
        .dropDuplicates(["gene_id_base"])
        .withColumn("importance", F.lit(None).cast("double"))
    )

genes_modelo = [r["gene_id_base"] for r in df_genes_ordenados.select("gene_id_base").collect()][:MAX_GENES_SPARKML]
print("Genes usados:", len(genes_modelo), genes_modelo[:10])

pdf_genes_modelo = pd.DataFrame({
    "orden": range(1, len(genes_modelo)+1),
    "gene_id_base": genes_modelo,
    "metodo_seleccion": "RFE_XGBoost_multiclase_local",
    "tabla_origen": "refined_rfe_feature_importances",
    "n_genes_total": len(genes_modelo)
})
save_spark_table(spark.createDataFrame(pdf_genes_modelo), "refined_rfe_genes_seleccionados_multiclase")

Genes cargados desde refined_rfe_feature_importances
Genes usados: 200 ['ENSG00000167751', 'ENSG00000128242', 'ENSG00000131095', 'ENSG00000109072', 'ENSG00000106927', 'ENSG00000042832', 'ENSG00000177807', 'ENSG00000077498', 'ENSG00000143167', 'ENSG00000198758']
Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_rfe_genes_seleccionados_multiclase


WindowsPath('C:/Users/mirol/Documents/AAcien/Integrador/notebooks_locales_tcga/notebooks_locales_tcga/data_local/refined/tables/refined_rfe_genes_seleccionados_multiclase')

In [8]:
df_long_modelo = (
    df_long
    .filter(F.col("gene_id_base").isin(genes_modelo))
    .select("sample_id", "patient_id", "cancer_type", "gene_id_base", "log2_tpm")
)

df_matriz = (
    df_long_modelo
    .groupBy("sample_id", "patient_id", "cancer_type")
    .pivot("gene_id_base", genes_modelo)
    .agg(F.first("log2_tpm"))
    .fillna(0.0)
)

columnas_base = ["sample_id", "patient_id", "cancer_type"]
columnas_genes = [c for c in genes_modelo if c in df_matriz.columns]

for c in columnas_genes:
    df_matriz = df_matriz.withColumn(c, F.coalesce(F.col(c).cast("double"), F.lit(0.0)))

df_matriz = df_matriz.select(*columnas_base, *columnas_genes)

print("Matriz:", df_matriz.count(), "filas x", len(df_matriz.columns), "columnas")
mostrar(df_matriz, 5)

save_spark_table(df_matriz, "refined_ml_matriz_rfe_200_genes")
save_spark_table(df_matriz, "refined_ml_matriz_modelado_desde_feature_selection")

Matriz: 8335 filas x 203 columnas


,sample_id,patient_id,cancer_type,ENSG00000167751,ENSG00000128242,ENSG00000131095,ENSG00000109072,ENSG00000106927,ENSG00000042832,ENSG00000177807,...,ENSG00000144339,ENSG00000165905,ENSG00000129354,ENSG00000117595,ENSG00000136160,ENSG00000086548,ENSG00000160182,ENSG00000169474,ENSG00000132437,ENSG00000158874
0,8bc46a09-7328-42e0-ad97-e557ec81048e,TCGA-A3-3365,KIRC,0.015926,6.161912,0.178619,1.127435,1.417974,0.368824,1.894178,...,0.100574,3.913665,3.605826,3.721876,8.491441,0.033934,0.363451,0.000000,4.970509,1.228111
1,2c9bfc03-d4e6-47be-a046-1e2efd086bf0,TCGA-B0-5080,KIRC,0.000000,6.219445,0.441271,0.794187,0.690641,0.616922,0.188654,...,0.000000,1.011853,1.493084,1.487538,5.722392,0.306146,0.000000,2.982218,1.857901,0.000000
2,68b5218b-b7f0-4549-af44-dfcbf327bc48,TCGA-B8-A54H,KIRC,0.000000,8.413495,0.615699,0.735609,1.364797,0.884637,0.285343,...,0.036890,5.156793,4.895201,4.022705,7.020936,0.738379,0.497536,0.000000,8.140321,0.000000
3,18714b14-6138-46af-99a1-24acbc326530,TCGA-B0-5092,KIRC,0.025596,7.742118,0.844305,0.309176,0.594357,0.561008,1.252839,...,0.013927,1.945870,3.677441,0.783247,5.302246,0.132511,1.183899,0.000000,4.869082,0.311271
4,3f733c5c-acbf-4e59-b8d9-c8f754970db3,TCGA-B2-5633,KIRC,0.030407,4.733278,0.657183,2.114933,0.944783,0.778461,0.720454,...,1.973464,1.611645,2.038822,3.574695,6.389155,1.299303,0.349705,0.000000,2.944109,1.279947


Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_ml_matriz_rfe_200_genes
Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_ml_matriz_modelado_desde_feature_selection


WindowsPath('C:/Users/mirol/Documents/AAcien/Integrador/notebooks_locales_tcga/notebooks_locales_tcga/data_local/refined/tables/refined_ml_matriz_modelado_desde_feature_selection')

In [9]:
# Split de pacientes desde refined_split_pacientes
df_split = load_spark_table("refined_split_pacientes").select("patient_id", "split")

df_matriz_split = df_matriz.join(df_split, on="patient_id", how="inner")
df_train = df_matriz_split.filter(F.col("split") == "train").drop("split")
df_val = df_matriz_split.filter(F.col("split") == "validation").drop("split")
df_test = df_matriz_split.filter(F.col("split") == "test").drop("split")

print("Train:", df_train.count(), "Validation:", df_val.count(), "Test:", df_test.count())
mostrar(df_train.groupBy("cancer_type").agg(F.count("*").alias("n")).orderBy(F.desc("n")), 30)

conteo_train = df_train.groupBy("cancer_type").agg(F.count("*").alias("n_clase"))
n_train = df_train.count()
n_clases = conteo_train.count()

df_pesos = (
    conteo_train
    .withColumn("class_weight", F.lit(n_train) / (F.lit(n_clases) * F.col("n_clase")))
    .select("cancer_type", "class_weight")
)

df_train = df_train.join(df_pesos, on="cancer_type", how="left").fillna({"class_weight": 1.0})
df_val = df_val.join(df_pesos, on="cancer_type", how="left").fillna({"class_weight": 1.0})
df_test = df_test.join(df_pesos, on="cancer_type", how="left").fillna({"class_weight": 1.0})

Train: 5827 Validation: 1249 Test: 1259


,cancer_type,n
0,BRCA,773
1,UCEC,384
2,KIRC,375
3,LUAD,368
4,HNSC,364
5,LGG,361
6,THCA,354
7,PRAD,351
8,LUSC,351
9,COAD,329


In [10]:
label_indexer_base = StringIndexer(inputCol="cancer_type", outputCol="label", handleInvalid="error")
assembler_base = VectorAssembler(inputCols=columnas_genes, outputCol="features_raw", handleInvalid="keep")
scaler_base = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True)

label_names_global = list(label_indexer_base.fit(df_train).labels)
print("Clases:", label_names_global)

def crear_pipeline(clf):
    return Pipeline(stages=[label_indexer_base, assembler_base, scaler_base, clf])

def grid_dict(**kwargs):
    keys = list(kwargs.keys())
    values = [kwargs[k] for k in keys]
    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]

def obtener_labels_pipeline(modelo):
    return list(modelo.stages[0].labels)

def predicciones_a_pandas(predicciones, labels):
    base = ["sample_id", "patient_id", "cancer_type", "label", "prediction"]
    if "probability" in predicciones.columns:
        pdf = predicciones.select(*base, vector_to_array("probability").alias("proba")).toPandas()
        probas = np.vstack(pdf["proba"].apply(lambda x: np.array(x, dtype=float)).values)
    else:
        pdf = predicciones.select(*base).toPandas()
        probas = None
    pdf["label"] = pdf["label"].astype(int)
    pdf["prediction"] = pdf["prediction"].astype(int)
    pdf["cancer_predicho"] = pdf["prediction"].apply(lambda i: labels[int(i)])
    return pdf, probas

def calcular_metricas(nombre_modelo, split, pred, labels):
    pdf, probas = predicciones_a_pandas(pred, labels)
    y_true, y_pred = pdf["label"].values, pdf["prediction"].values
    clases = list(range(len(labels)))
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, labels=clases, average="macro", zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, labels=clases, average="weighted", zero_division=0)
    roc = pr = np.nan
    if probas is not None and probas.shape[1] == len(clases):
        try:
            roc = roc_auc_score(y_true, probas, labels=clases, multi_class="ovr", average="macro")
        except Exception:
            pass
        try:
            pr = average_precision_score(label_binarize(y_true, classes=clases), probas, average="macro")
        except Exception:
            pass
    return {
        "modelo": nombre_modelo, "split": split,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(p_macro), "recall_macro": float(r_macro), "f1_macro": float(f1_macro),
        "precision_weighted": float(p_w), "recall_weighted": float(r_w), "f1_weighted": float(f1_w),
        "roc_auc_macro_ovr": None if np.isnan(roc) else float(roc),
        "pr_auc_macro_ovr": None if np.isnan(pr) else float(pr),
    }

Clases: ['BRCA', 'UCEC', 'KIRC', 'LUAD', 'HNSC', 'LGG', 'THCA', 'LUSC', 'PRAD', 'COAD', 'OV', 'STAD', 'BLCA', 'LIHC', 'CESC', 'KIRP', 'GBM', 'SKCM']


In [11]:
# Factories de modelos SparkML

def factory_logistic_regression(params):
    return LogisticRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        weightCol="class_weight",
        family="multinomial",
        maxIter=params["maxIter"],
        regParam=params["regParam"],
        elasticNetParam=params["elasticNetParam"]
    )


def factory_xgboost(params):
    return SparkXGBClassifier(
        features_col="features",
        label_col="label",
        prediction_col="prediction",
        probability_col="probability",
        raw_prediction_col="rawPrediction",
        weight_col="class_weight",
        num_workers=1,
        coll_cfg=XGBCollectiveConfig(tracker_host_ip="127.0.0.1"),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=SEED,
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=params["min_child_weight"],
        gamma=params["gamma"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"]
    )


def factory_one_vs_rest_linear_svc(params):
    base_svc = LinearSVC(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        rawPredictionCol="rawPrediction",
        weightCol="class_weight",
        maxIter=params["maxIter"],
        regParam=params["regParam"]
    )

    return OneVsRest(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        classifier=base_svc
    )


## Ajuste de hiperparámetros con Optuna (los 3 modelos, 5 trials c/u)

Cada modelo (Regresión Logística, SVM lineal y XGBoost) se optimiza por separado con Optuna usando validación cruzada **agrupada por paciente** y `f1_macro`. Por defecto **5 trials por modelo (15 en total)**, configurable con `TCGA_N_TRIALS`. Luego cada modelo se reentrena con sus mejores hiperparámetros y se evalúa en train/validation/test; el de mayor `f1_macro` en CV es el final.

In [12]:
# === Optuna para los 3 modelos (5 trials c/u por defecto) ===
import os
import numpy as np
import optuna
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = int(os.getenv("TCGA_N_TRIALS", "5"))   # por modelo -> 15 en total
N_FOLDS  = int(os.getenv("TCGA_N_FOLDS", "5"))

def _xy(sdf):
    pdf = sdf.select("sample_id", "patient_id", "cancer_type", *columnas_genes).toPandas()
    X = pdf[columnas_genes].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy("float32")
    return pdf, X

le = LabelEncoder()
pdf_tr, Xtr = _xy(df_train); ytr = le.fit_transform(pdf_tr["cancer_type"])
pdf_va, Xva = _xy(df_val);   yva = le.transform(pdf_va["cancer_type"])
pdf_te, Xte = _xy(df_test);  yte = le.transform(pdf_te["cancer_type"])
groups_tr = pdf_tr["patient_id"].to_numpy()

cv = list(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(Xtr, ytr, groups_tr))

def _cv_f1(make_model):
    sc = []
    for tr_idx, va_idx in cv:
        m = make_model()
        sw = compute_sample_weight("balanced", ytr[tr_idx])
        try:
            m.fit(Xtr[tr_idx], ytr[tr_idx], sample_weight=sw)
        except TypeError:
            m.fit(Xtr[tr_idx], ytr[tr_idx])
        sc.append(f1_score(ytr[va_idx], m.predict(Xtr[va_idx]), average="macro"))
    return float(np.mean(sc))

def _obj_logreg(t):
    C = t.suggest_float("C", 1e-2, 10.0, log=True)
    return _cv_f1(lambda: LogisticRegression(C=C, max_iter=2000, multi_class="multinomial", n_jobs=-1))

def _obj_svc(t):
    C = t.suggest_float("C", 1e-2, 10.0, log=True)
    return _cv_f1(lambda: LinearSVC(C=C, max_iter=5000))

def _obj_xgb(t):
    p = dict(
        max_depth=t.suggest_int("max_depth", 3, 8),
        learning_rate=t.suggest_float("learning_rate", 1e-2, 0.3, log=True),
        n_estimators=t.suggest_int("n_estimators", 100, 400),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
    )
    return _cv_f1(lambda: XGBClassifier(objective="multi:softprob", eval_metric="mlogloss",
                                        tree_method="hist", random_state=SEED, n_jobs=-1, **p))

ESPACIOS = {
    "LogisticRegression": (_obj_logreg, lambda bp: LogisticRegression(max_iter=2000, multi_class="multinomial", n_jobs=-1, **bp)),
    "SVM_LinearSVC":      (_obj_svc,    lambda bp: LinearSVC(max_iter=5000, **bp)),
    "XGBoost":            (_obj_xgb,    lambda bp: XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", random_state=SEED, n_jobs=-1, **bp)),
}

estudios = {}
for nombre, (objetivo, _) in ESPACIOS.items():
    print(f"Optuna -> {nombre} ({N_TRIALS} trials)")
    est = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    est.optimize(objetivo, n_trials=N_TRIALS)
    estudios[nombre] = est
    print(f"   mejor f1_macro CV = {est.best_value:.4f} | params = {est.best_params}")

Optuna -> LogisticRegression (5 trials)


C:\Users\mirol\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\mirol\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\mirol\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\mirol\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_mo

   mejor f1_macro CV = 0.9600 | params = {'C': 0.02938027938703535}
Optuna -> SVM_LinearSVC (5 trials)
   mejor f1_macro CV = 0.9559 | params = {'C': 0.02938027938703535}
Optuna -> XGBoost (5 trials)
   mejor f1_macro CV = 0.9647 | params = {'max_depth': 5, 'learning_rate': 0.14447746112718687, 'n_estimators': 160, 'subsample': 0.8056937753654446, 'colsample_bytree': 0.836965827544817, 'reg_lambda': 0.013783237455007183}


In [13]:
# === Refit de cada modelo y evaluacion en train/validation/test ===
from sklearn.metrics import accuracy_score, balanced_accuracy_score

sw_full = compute_sample_weight("balanced", ytr)
filas, modelos_fit = [], {}
for nombre, (_, constructor) in ESPACIOS.items():
    m = constructor(estudios[nombre].best_params)
    try:
        m.fit(Xtr, ytr, sample_weight=sw_full)
    except TypeError:
        m.fit(Xtr, ytr)
    modelos_fit[nombre] = m
    for split, X, y in [("train", Xtr, ytr), ("validation", Xva, yva), ("test", Xte, yte)]:
        yp = m.predict(X)
        filas.append({"modelo": nombre, "split": split,
                      "accuracy": float(accuracy_score(y, yp)),
                      "balanced_accuracy": float(balanced_accuracy_score(y, yp)),
                      "f1_macro": float(f1_score(y, yp, average="macro"))})

pdf_metricas = pd.DataFrame(filas)
display(pdf_metricas.sort_values(["split", "f1_macro"], ascending=[True, False]))
save_spark_table(spark.createDataFrame(pdf_metricas), "refined_metricas_modelos_sparkml")

nombre_final = max(ESPACIOS, key=lambda n: estudios[n].best_value)
modelo_final = modelos_fit[nombre_final]
print("Modelo final (mejor CV):", nombre_final)

pdf_final = pdf_metricas[pdf_metricas["modelo"] == nombre_final].copy()
save_spark_table(spark.createDataFrame(pdf_final), "refined_metricas_modelo_final")

pdf_hiper = pd.DataFrame([{"modelo": nombre_final, "parametro": k, "valor": str(v)}
                          for k, v in estudios[nombre_final].best_params.items()])
if not pdf_hiper.empty:
    save_spark_table(spark.createDataFrame(pdf_hiper), "refined_hiperparametros_modelos_sparkml")

import joblib
joblib.dump(modelo_final, MODELS_PATH / "modelo_final.joblib")
joblib.dump(le, MODELS_PATH / "label_encoder_modelo_final.joblib")

C:\Users\mirol\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,modelo,split,accuracy,balanced_accuracy,f1_macro
8,XGBoost,test,0.971406,0.972413,0.969830
2,LogisticRegression,test,0.969023,0.966964,0.965791
5,SVM_LinearSVC,test,0.962669,0.960259,0.959176
6,XGBoost,train,1.000000,1.000000,1.000000
3,SVM_LinearSVC,train,0.997597,0.997826,0.997660
0,LogisticRegression,train,0.994337,0.994636,0.993995
7,XGBoost,validation,0.972778,0.972055,0.971976
1,LogisticRegression,validation,0.972778,0.970669,0.968895
4,SVM_LinearSVC,validation,0.971177,0.969663,0.968061


Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_metricas_modelos_sparkml
Modelo final (mejor CV): XGBoost
Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_metricas_modelo_final
Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_hiperparametros_modelos_sparkml


['C:\\Users\\mirol\\Documents\\AAcien\\Integrador\\notebooks_locales_tcga\\notebooks_locales_tcga\\data_local\\models\\label_encoder_modelo_final.joblib']

## Diagnóstico del modelo final (test)

Matriz de confusión, reporte por clase y predicciones del modelo final. Alimenta el notebook 06.

In [14]:
from sklearn.metrics import classification_report, confusion_matrix

clases = list(le.classes_)
idx = list(range(len(clases)))
y_pred_te = modelo_final.predict(Xte)

reporte = classification_report(yte, y_pred_te, labels=idx, target_names=clases, output_dict=True, zero_division=0)
pdf_reporte = pd.DataFrame(reporte).transpose().reset_index().rename(columns={"index": "clase"})
display(pdf_reporte)
save_spark_table(spark.createDataFrame(pdf_reporte), "refined_reporte_clasificacion_por_clase")

cm = confusion_matrix(yte, y_pred_te, labels=idx)
pdf_cm = pd.DataFrame([{"true_class": clases[i], "predicted_class": clases[j], "n": int(cm[i, j])}
                      for i in range(len(clases)) for j in range(len(clases))])
save_spark_table(spark.createDataFrame(pdf_cm), "refined_matriz_confusion_mejor_modelo")

pdf_pred = pdf_te[["sample_id", "patient_id", "cancer_type"]].copy()
pdf_pred["label"] = yte
pdf_pred["prediction"] = y_pred_te
pdf_pred["cancer_predicho"] = le.inverse_transform(y_pred_te)
pdf_pred["modelo"] = nombre_final
save_spark_table(spark.createDataFrame(pdf_pred), "refined_predicciones_test_mejor_modelo")
print("Diagnostico guardado.")

,clase,precision,recall,f1-score,support
0,BLCA,0.966667,0.906250,0.935484,64.000000
1,BRCA,1.000000,0.987952,0.993939,166.000000
2,CESC,0.913043,0.933333,0.923077,45.000000
3,COAD,1.000000,0.972603,0.986111,73.000000
4,GBM,0.934783,1.000000,0.966292,43.000000
5,HNSC,0.905882,0.987179,0.944785,78.000000
6,KIRC,1.000000,0.950617,0.974684,81.000000
7,KIRP,0.914894,1.000000,0.955556,43.000000
8,LGG,1.000000,0.961538,0.980392,78.000000
9,LIHC,0.982143,1.000000,0.990991,55.000000


Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_reporte_clasificacion_por_clase
Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_matriz_confusion_mejor_modelo
Tabla local guardada: C:\Users\mirol\Documents\AAcien\Integrador\notebooks_locales_tcga\notebooks_locales_tcga\data_local\refined\tables\refined_predicciones_test_mejor_modelo
Diagnostico guardado.
